In [3]:
#Ishaan
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
df = pd.read_csv("../data/nodules_labels.csv")
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
SOFT_COLS = [f"soft_{k}" for k in range(1, 6)]

In [4]:
#train; input: the size, as diameter
x = train["diameter_mm"].to_numpy()     # plain numbers, shape (1305,)
x_train = torch.tensor(x, dtype=torch.float32)[:, None]  # shape (1305, 1)
# HARD target: the median rating as a class number 0-4
y_train = torch.tensor(train["hard"].to_numpy() - 1, dtype=torch.int64)   # shape (1305,)
# SOFT target: the share of readers who gave each rating, rows sum to 1
y_train_soft = torch.tensor(train[SOFT_COLS].to_numpy(), dtype=torch.float32)  # shape (1305, 5)

#val
x = val["diameter_mm"].to_numpy()
x_val = torch.tensor(x,dtype=torch.float32)[:,None]
y_val = torch.tensor(val["hard"].to_numpy()-1, dtype=torch.int64)
y_val_soft = torch.tensor(val[SOFT_COLS].to_numpy(), dtype=torch.float32)

In [5]:
def train_model(x, target, seed=0):
    """Same model, same optimiser, same steps. The ONLY difference between the two models is `target`."""
    torch.manual_seed(seed)        # both models start from identical weights
    model = nn.Linear(1,5)
    loss_fn = nn.CrossEntropyLoss() # takes class numbers (hard) OR 5 probabilities per nodule (soft)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.05) #adjusts the model's 10 numbers (5 weights and 5 biases) to make the loss smaller. learning rate=0.05 is how big each adjustment is.
    for step in range(500):
        optimiser.zero_grad()           # 1. clear the previous step's gradients
        out = model(x)                  # 2. forward: 5 scores per nodule, shape (1305, 5)
        loss = loss_fn(out, target)     # 3. how wrong is it?
        loss.backward()                 # 4. work out which way to nudge each number
        optimiser.step()                # 5. nudge them
        if step % 100 == 0:
            print(step, round(loss.item(), 4))
    return model

print("HARD model (target = median rating)")
hard_model = train_model(x_train, y_train)
print("SOFT model (target = readers' distribution)")
soft_model = train_model(x_train, y_train_soft)


HARD model (target = median rating)
0 10.3446
100 1.239
200 1.1819
300 1.1655
400 1.1603
SOFT model (target = readers' distribution)
0 9.1713
100 1.3846
200 1.3528
300 1.3497
400 1.3494
lowest possible soft loss: 0.6181


In [7]:
import sys
sys.path.insert(0, "/Users/Rai/Downloads/Classifying-malignancy-in-lung-nodules-using-deep-learning-models/pipeline")
import metrics as M

hard = val["hard"].to_numpy()
soft = val[SOFT_COLS].to_numpy()

# BOTH models output 5 probabilities per nodule
probs = {}
results = {}
for name, model in [("hard", hard_model), ("soft", soft_model)]:
    with torch.no_grad():                                          # scoring only, no learning
        probs[name] = torch.softmax(model(x_val), dim=1).numpy()  # (290, 5), each row sums to 1
    results[name] = M.all_metrics(probs[name], hard, soft)         # 4 headline numbers + quadratic kappa

print(pd.DataFrame(results).round(3))

                    hard   soft
balanced_accuracy  0.360  0.319
auc                0.913  0.913
ce_vs_readers      1.432  1.351
ece                0.023  0.135
quadratic_kappa    0.423  0.347


nodule LIDC-IDRI-0006_920957_02: 8.3 mm, readers rated 3;4;3;2
   readers      [0.   0.25 0.5  0.25 0.  ]
   hard model   [0.11 0.16 0.57 0.13 0.02] -> top rating 3 (57% sure)
   soft model   [0.16 0.24 0.43 0.12 0.04] -> top rating 3 (43% sure)
nodule LIDC-IDRI-0012_457842_02: 11.9 mm, readers rated 4;3;5;2
   readers      [0.   0.25 0.25 0.25 0.25]
   hard model   [0.06 0.09 0.52 0.26 0.07] -> top rating 3 (52% sure)
   soft model   [0.11 0.21 0.43 0.18 0.08] -> top rating 3 (43% sure)
nodule LIDC-IDRI-0012_457842_04: 7.2 mm, readers rated 3;3;4;2
   readers      [0.   0.25 0.5  0.25 0.  ]
   hard model   [0.13 0.18 0.56 0.1  0.02] -> top rating 3 (56% sure)
   soft model   [0.18 0.25 0.43 0.11 0.03] -> top rating 3 (43% sure)
